# Project: UrbanCart AI Shopping Assistant — RAG-Powered Product & Policy Q&A Engine
    - **Domain:** E-Commerce (Fashion, Electronics & Home Goods) — Mobile App AI Backend
    - **Type:** RAG-based GenAI System + Lightweight Agentic Tool-Calling

---

## Business Problem Statement

You are a **Senior Mobile Application Developer** at **UrbanCart**, a mid-sized Indian e-commerce marketplace app (fashion, electronics, and home & kitchen categories) with **2M+ monthly active users**. Over the last several release cycles you have owned the iOS app end-to-end and the **Node.js/NestJS API gateway** and **AWS Lambda** services that the app talks to — including Firebase push notifications, order APIs, and the in-app chat surface.

UrbanCart's in-app and call-center support handles roughly **6,000 customer conversations per day**. Internal analysis by the CX team shows that **~40% of these are repetitive, answerable-from-documentation questions**: *"Is this kurti available in size XL?"*, *"Does this earphone support wireless charging?"*, *"What's the return window on electronics vs apparel?"*, *"Is COD available for my pincode?"*. The existing in-app FAQ bot is a brittle keyword-matcher — it cannot handle comparisons, partial product names, Hinglish phrasing, or any question that combines a product attribute with a policy rule (e.g., *"can I return this if I bought it on COD?"*). App store reviews increasingly mention the bot being "useless," and the CX team is missing its first-response-time SLA during sale events.

Leadership has greenlit a new **"Ask UrbanCart"** screen inside the mobile app, backed by a **Python AI microservice** that your existing NestJS gateway can call over a simple REST contract — a natural extension of the architecture you already own, not a rip-and-replace. The mandate is explicit: answers must be **grounded in the actual product catalog and policy documents** (no hallucinated prices, stock, or return rules), the assistant should be able to **look up live-ish data** (stock status, order status) rather than only answer from static text, and anything it can't confidently resolve must be **escalated to a human agent with a clean summary** rather than guessed at.

**Your mission:** Build a **RAG-powered Product & Policy Q&A engine** with a thin agentic tool-calling layer that:
1. Answers product spec, comparison, and policy questions grounded in the catalog and policy documents (not hallucinated)
2. Uses simple tools to check stock status, look up order status, and recommend similar products
3. Detects Hinglish, adversarial, and off-topic queries and handles each appropriately
4. Ships as a deployable microservice + stakeholder demo that your Node.js/NestJS gateway and the product team can actually integrate with

---

> **What this teaches you for your job:**
> Product + policy RAG assistants are one of the **most common GenAI use cases in Indian e-commerce** (Myntra, Flipkart, Meesho, Ajio all run variants of this). As a mobile/backend engineer, owning the *integration* of a GenAI microservice into an app architecture you already maintain is a uniquely strong hiring story — most candidates can only show the AI half; you can show the AI half **and** the production path it has to travel through (gateway → microservice → mobile UI). This project is the RAG foundation; later projects in this series add agentic orchestration and full LLMOps deployment on top of the same UrbanCart system.

---

## System Architecture

```
┌──────────────────────────────────────────────────────────────────────┐
│                         URBANCART "ASK AI" ASSISTANT                 │
│                                                                      │
│  Mobile App (iOS)                                                    │
│  "Ask UrbanCart" screen                                              │
│        │ REST call                                                   │
│        ▼                                                            │
│  NestJS API Gateway  (existing infra you own)                        │
│        │ proxies request                                             │
│        ▼                                                            │
│  Flask AI Microservice  (built in this project)                    │
│        │                                                             │
│        ▼                                                            │
│   Query Router  ──► [intent: product_qa / policy_qa / lookup / off]  │
│        │                          │                    │             │
│        ▼                          ▼                    ▼             │
│   RAG Chain                 Tool-Calling Agent     Escalation        │
│   (catalog + policy)        (stock / order /        Handler         │
│        │                     similar-product)         │              │
│        ▼                          │                    ▼             │
│   Vector Store               Mock LOS / Catalog    Human Queue       │
│   (FAISS / Chroma)              DB lookups                           │
│        │                                                              │
│        ▼                                                              │
│   LLM + Context ──► Structured Answer ──► back through gateway ──►   │
│                                                       Mobile App UI   │
└──────────────────────────────────────────────────────────────────────┘
```

### Business KPIs You'll Measure
- **Answer Faithfulness**: % of RAG answers correctly grounded in source documents
- **Hallucination Rate**: % of answers containing info NOT in catalog/policy docs
- **Latency**: Average response time (target: <3 seconds)
- **Escalation Rate**: % of queries routed to a human agent (target: <15%)
- **Cost per Query**: Average LLM API cost (target: <₹0.50 per query)
- **Deflection Rate**: % of repetitive support volume the assistant resolves without a human

---

## Stage 0: Environment Setup & Prerequisites

### API Keys & Tools Required

| Tool | Purpose | Where to Get | Free Tier? |
|------|---------|--------------|------------|
| OpenAI API | LLM (GPT-3.5/GPT-4) | platform.openai.com | $5 free credit |
| HuggingFace | Open-source / multilingual embeddings | huggingface.co/settings/tokens | Free |
| FAISS or ChromaDB | Local vector store (no account needed) | pip install | Free |
| LangSmith | LLM observability/tracing | smith.langchain.com | Free tier |
| Gradio | Stakeholder demo UI | pip install | Free |

### Cost Warning
- `gpt-3.5-turbo`: ~₹0.05–0.15 per query (affordable for dev/testing)
- `gpt-4` / `gpt-4o`: ~10–20x the cost — reserve for evaluation/LLM-as-judge only
- **Free alternative:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` for embeddings (handles Hindi + English) avoids embedding API costs entirely

### Compute Requirements
- Google Colab Free (T4 GPU) is sufficient for every stage in this project
- RAM: 12GB standard Colab tier is enough
- No GPU is strictly required — embeddings and LLM calls are API-based by default

### Package Installation

In [1]:
# Stage 0: Install all required packages

# Run once in a fresh environment, then restart the kernel.
# %pip install -r requirements.txt
print("Dependencies are listed in requirements.txt.")


Dependencies are listed in requirements.txt.


In [2]:
# Stage 0: API Key Configuration

import json
import os
from pathlib import Path
from IPython.display import display, HTML
import pandas as pd
from urbancart.config import RAW, PROCESSED, Settings

# OpenRouter and optional LangSmith credentials are loaded from the local .env.
settings = Settings.from_env(mode="openrouter")
print("OpenRouter key configured:", bool(settings.key))


OpenRouter key configured: True


---
## Stage 1: Architecture Decisions & Project Configuration

> **Industry Context:**
> At companies like Myntra and Flipkart, before any RAG system touches production, the engineering team documents *why* each architectural choice was made — embedding model, chunk size, vector store — because these decisions directly affect retrieval quality, latency, and monthly API cost at scale. Interviewers will ask you to justify these numbers, not just state them.

### Your Task — Configuration with Justification

In [3]:
# Stage 1: Define project configuration and constants

PROJECT_CONFIG = {
    "project_name": "UrbanCart Ask-AI Assistant",
    "version": "1.0.0",
    "embedding_model": settings.embedding_model,
    "llm_model": settings.chat_model,
    "vector_db": "chroma",
    "product_chunk_size": "One complete record per SKU",
    "policy_chunk_size": "One numbered clause",
    "chunk_overlap": 0,
    "top_k_retrieval": 5,
    "max_escalation_threshold": 0.6,
}
# Short source records already have natural boundaries. Related policy
# clauses are added together so material conditions reach the model.
# Chroma persists the index. Semantic search complements exact SKU filters.
display(pd.Series(PROJECT_CONFIG))


project_name                   UrbanCart Ask-AI Assistant
version                                             1.0.0
embedding_model             openai/text-embedding-3-small
llm_model                    google/gemini-3.1-flash-lite
vector_db                                          chroma
product_chunk_size            One complete record per SKU
policy_chunk_size                     One numbered clause
chunk_overlap                                           0
top_k_retrieval                                         5
max_escalation_threshold                              0.6
dtype: object

---
## Stage 3: Exploratory Data Analysis (EDA) & Data Profiling

> **Industry Context:**
> Before any product/catalog data feeds a customer-facing AI feature, the **Data Quality** and **Catalog Ops** teams require a documented profiling pass — a wrong price or stock status surfaced confidently by an AI assistant is a direct trust and compliance problem (Consumer Protection Act / E-Commerce Rules in India require accurate pricing disclosure). Your EDA here is that documentation.

### EDA Objectives
1. Quantify data quality issues (missing, duplicate, format inconsistency) in the catalog
2. Understand distribution of prices, ratings, categories, stock status
3. Identify business rule violations
4. Create a Data Quality Score for the catalog
5. Profile the chat query log for intent/language distribution (needed for Stage 5)

In [4]:
# Stage 3A: Basic Data Profiling — Product Catalog

from urbancart.data import CATALOG, CATEGORIES, amount, normalize, price_style, read_source, value

df_catalog = read_source(RAW / CATALOG)
print("Catalog shape:", df_catalog.shape)
df_catalog.info()

missing = df_catalog.apply(lambda col: col.map(value).isna())
missing_summary = pd.DataFrame({"Missing": missing.sum(), "% missing": 100 * missing.mean()})
print("Total missing cells:", int(missing.sum().sum()))
display(missing_summary.style.format({"% missing": "{:.2f}"}).bar(subset=["% missing"], color="#ffb36b"))
display(missing_summary[missing_summary["% missing"] > 30])
# Aggregate all input rows into consecutive groups for a readable heatmap.
heatmap = missing.groupby(missing.index // 100).mean() * 100
display(heatmap.style.format("{:.0f}").apply(
    lambda col: [f"background: rgba(234, 88, 12, {x / 100:.2f})" for x in col]
).set_caption("Missing values (%) by source block of 100 rows"))

duplicate_skus = df_catalog[df_catalog["sku"].duplicated(keep=False)]
print("Exact duplicate rows:", df_catalog.duplicated().sum())
print("Repeated SKUs:", duplicate_skus["sku"].nunique())
conflicts = duplicate_skus.assign(parsed_price=duplicate_skus["price"].map(amount))
counts = conflicts.groupby("sku")[["parsed_price", "stock_quantity"]].nunique()
print("Repeated SKUs with differing observed prices or stock:", counts.gt(1).any(axis=1).sum())

def distribution(series, title):
    table = series.value_counts(dropna=False).rename("count").to_frame()
    display(table.style.bar(subset=["count"], color="#8ac6e8").set_caption(title))

distribution(df_catalog["price"].map(price_style), "Price formats")
prices = df_catalog["price"].map(amount).div(100)
distribution(pd.cut(prices, [0, 500, 2000, 5000, 10000, float("inf")]), "Parsed selling prices (INR)")
ratings = pd.to_numeric(df_catalog["rating"], errors="coerce")
rating_flags = pd.Series("out_of_range", index=ratings.index)
rating_flags[ratings.isna()] = "invalid_text"
rating_flags[ratings.between(1, 5)] = "valid_numeric"
rating_flags[missing["rating"]] = "null"
distribution(rating_flags, "Rating validity")
category_map = df_catalog["category"].str.strip().str.casefold().map(CATEGORIES)
category_group = category_map.replace({"Beauty & Personal Care": "Other"})
display(pd.DataFrame({"raw": df_catalog["category"], "group": category_group.fillna("Unknown")})
        .value_counts().rename("rows").to_frame().style.bar(subset=["rows"], color="#8ac6e8"))
stock = pd.to_numeric(df_catalog["stock_quantity"], errors="coerce")
distribution(pd.cut(stock, [-float("inf"), -1, 0, 10, float("inf")]),
             "Source stock quantity bands. Blank or invalid values appear as NaN.")


Catalog shape: (5150, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5150 entries, 0 to 5149
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   sku                 5150 non-null   object
 1   product_name        5150 non-null   object
 2   category            5150 non-null   object
 3   brand               5150 non-null   object
 4   price               5150 non-null   object
 5   mrp                 5150 non-null   object
 6   discount_pct        5150 non-null   object
 7   rating              5150 non-null   object
 8   stock_quantity      5150 non-null   object
 9   return_window_days  5150 non-null   object
 10  description         5150 non-null   object
 11  seller_id           5150 non-null   object
 12  date_added          5150 non-null   object
dtypes: object(13)
memory usage: 523.2+ KB
Total missing cells: 3747


,Missing,% missing
sku,0,0.00
product_name,0,0.00
category,312,6.06
brand,1405,27.28
price,1357,26.35
mrp,0,0.00
discount_pct,0,0.00
rating,0,0.00
stock_quantity,0,0.00
return_window_days,260,5.05


,Missing,% missing


,sku,product_name,category,brand,price,mrp,discount_pct,rating,stock_quantity,return_window_days,description,seller_id,date_added
0,0,0,6,24,25,0,0,0,0,5,6,0,0
1,0,0,6,26,26,0,0,0,0,6,7,0,0
2,0,0,7,24,29,0,0,0,0,7,9,0,0
3,0,0,7,26,26,0,0,0,0,4,9,0,0
4,0,0,4,23,24,0,0,0,0,4,7,0,0
5,0,0,5,22,24,0,0,0,0,4,7,0,0
6,0,0,6,26,21,0,0,0,0,5,9,0,0
7,0,0,10,34,25,0,0,0,0,9,12,0,0
8,0,0,4,24,25,0,0,0,0,2,5,0,0
9,0,0,7,26,28,0,0,0,0,6,10,0,0


Exact duplicate rows: 0
Repeated SKUs: 154
Repeated SKUs with differing observed prices or stock: 154


,count
price,
plain numeric,1378
missing,1357
currency prefix,667
k suffix,647
slash suffix,621
comma grouped,480


,count
price,
nan,1357
"(500.0, 2000.0]",1321
"(2000.0, 5000.0]",1315
"(10000.0, inf]",565
"(0.0, 500.0]",328
"(5000.0, 10000.0]",264


,count
valid_numeric,4912
out_of_range,238


,,rows
raw,group,
,Unknown,312
Personal Care,Other,271
Beauty & Grooming,Other,267
Shoes,Footwear,266
FOOTWEAR,Footwear,263
Footwear,Footwear,259
footwear,Footwear,247
Beauty,Other,237
beauty,Other,220


,count
stock_quantity,
"(10.0, inf]",4887
"(-inf, -1.0]",152
"(0.0, 10.0]",100
"(-1.0, 0.0]",11


In [5]:
# Stage 3B: Business Rule Violation Detection

BUSINESS_RULES = {
    'price_positive': 'Price must be a positive number',
    'discount_range': 'Discount % must be between 0 and 80',
    'rating_range': 'Valid rating: 1.0-5.0',
    'stock_quantity_non_negative': 'Stock quantity cannot be negative',
    'mrp_gte_price': 'MRP must be >= selling price',
    'return_window_valid': 'Return window must be within policy-allowed range (0-30 days)',
    'duplicate_sku': 'SKU must be unique in the active catalog',
}

from urbancart.quality import quality_report

def check_business_rules(df):
    from urbancart.quality import raw_catalog_quality
    return raw_catalog_quality(df)

raw_quality = check_business_rules(df_catalog)
display(pd.Series(raw_quality["failed_or_unverifiable_by_rule"], name="Failed or unverifiable"))
print(f'Raw DQ: {raw_quality["clean_records"]}/{raw_quality["records"]} = {raw_quality["dq_score_pct"]:.2f}%')


required_fields          1591
price_positive           1357
discount_range           1104
rating_range              238
stock_nonnegative         152
mrp_gte_price            2050
return_window_0_to_30     450
unique_sku                308
Name: Failed or unverifiable, dtype: int64

Raw DQ: 2330/5150 = 45.24%


In [6]:
# Stage 3C: Chat Query Log Profiling (needed for Stage 5 intent classification design)

from urbancart.data import CHAT

df_chats = read_source(RAW / CHAT)
distribution(df_chats["true_intent"], "Chat intents")
distribution(df_chats["language_mix"], "Chat languages")
resolved = df_chats["resolved_by_bot_currently"].str.strip().str.casefold().eq("true")
display((100 * resolved.groupby(df_chats["true_intent"]).mean()).rename("Previous bot resolution (%)"))
print(f"Previous bot baseline: {resolved.sum()}/{len(resolved)} ({resolved.mean():.1%})")


,count
true_intent,
product_qa,770
policy_qa,568
order_lookup,389
off_topic,166
adversarial,107


,count
language_mix,
en,1086
hinglish,628
hi,286


true_intent
adversarial      3.738318
off_topic       11.445783
order_lookup    30.077121
policy_qa       58.802817
product_qa      53.506494
Name: Previous bot resolution (%), dtype: float64

Previous bot baseline: 886/2000 (44.3%)


---
## Stage 4: Data Preprocessing & Cleaning

> **Industry Context:**
> At UrbanCart, the Catalog Engineering team runs this kind of cleaning pipeline as a **daily batch job** so the RAG index always reflects current prices/stock. Every transformation is logged — if a price is auto-corrected and that correction is wrong, the audit trail is what lets the team trace and fix it before it reaches a customer.

> **A common mistake:** Dropping rows with messy data. In e-commerce, dropping a product record means it silently disappears from the AI assistant's knowledge — a customer asks about a real product and gets "not found." Industry practice is to **impute with business logic, flag the imputation, and route heavily-imputed records for manual catalog review.**

In [7]:
# Stage 4A: Build the Data Cleaning Pipeline

from urbancart.data import number, prepare, read_products

def clean_price(price_value):
    paise = amount(price_value)
    if paise is None:
        return None, False, "Missing or unparseable source price"
    rupees = paise / 100
    clean = int(rupees) if paise % 100 == 0 else rupees
    return clean, str(clean) != str(price_value).strip(), "Parsed source price in INR"

def standardize_category(category_value):
    category = CATEGORIES.get(str(category_value).strip().casefold())
    return "Other" if category == "Beauty & Personal Care" else category

def standardize_stock_status(stock_value, stock_quantity=None):
    # The supplied file has no free-text stock-status field.
    qty = number(stock_quantity, 0, 10**9, True)
    if qty is None:
        return "Unknown"
    if qty == 0:
        return "Out of Stock"
    return "Low Stock" if qty <= 10 else "In Stock"

def clean_rating(rating_value):
    return number(rating_value, 1, 5)

def run_cleaning_pipeline():
    # Reads the same raw catalog/chat files as Stage 3 and writes the shared artifacts.
    profile = prepare()
    clean = pd.DataFrame(read_products())
    clean["stock_status"] = clean["stock"].map(lambda qty: standardize_stock_status(None, qty))
    clean["category_group"] = clean["category"].map(standardize_category)
    audit = pd.read_csv(PROCESSED / "cleaning_audit.csv", keep_default_na=False)
    return clean, audit, profile

df_clean, df_audit, profile = run_cleaning_pipeline()
display(df_clean[["sku", "name", "category_group", "price_paise", "stock", "stock_status", "rating"]].head())
display(df_audit.head())
review = pd.read_csv(PROCESSED / "manual_review.csv")
print(f'{profile["catalog_rows"]} source rows retained across {len(df_clean)} SKUs.')
print(f'{review["needs_manual_review"].sum()} source rows flagged for manual review.')
complete = df_clean[["price_paise", "stock", "category"]].notna().all(axis=1)
print(f"Checkpoint 1 required-field coverage: {complete.sum()}/{len(complete)} ({complete.mean():.2%}); target >90%.")
print(f'Overall cleaned business-rule DQ: {quality_report()["resolved_skus"]["dq_score_pct"]:.2f}%.')


,sku,name,category_group,price_paise,stock,stock_status,rating
0,UC-ELE-916693,Wireless Mouse,Electronics,NaN,283.0,In Stock,NaN
1,UC-ELE-891071,LED Monitor,Electronics,3335000.0,451.0,In Stock,1.7
2,UC-HOM-957955,Mixer Grinder,Home & Kitchen,NaN,455.0,In Stock,4.1
3,UC-FAS-366422,Formal Shirt,Fashion,192000.0,NaN,Unknown,4.9
4,UC-HOM-325889,Steel Water Bottle,Home & Kitchen,132000.0,19.0,In Stock,3.5


,source_row,sku,field,raw,clean,rule
0,2,UC-ELE-916693,category,electronics,Electronics,normalized
1,2,UC-ELE-916693,price_paise,Rs 29560/-,,price_above_mrp
2,2,UC-ELE-916693,mrp_paise,26604,,price_pair_conflict
3,2,UC-ELE-916693,discount_pct,-11,,price_pair_conflict
4,2,UC-ELE-916693,rating,0.0,,invalid_or_unreliable


5150 source rows retained across 4996 SKUs.
1932 source rows flagged for manual review.
Checkpoint 1 required-field coverage: 2850/4996 (57.05%); target >90%.


Overall cleaned business-rule DQ: 49.62%.


---
## Checkpoint 1 — Data Foundation

Before moving to the GenAI stages, verify:
- [ ] Completed EDA with missing value analysis and business rule violation detection
- [ ] Built a cleaning pipeline with audit logging
- [ ] Data Quality Score improved from baseline to >90% for required fields (price, stock, category)


**Industry Benchmark:** Production e-commerce catalog pipelines typically target 95%+ completeness on Tier-1 fields (price, stock, category) before any downstream system — search, recommendations, or AI — consumes the data.

---

## Stage 5: LLM Setup & Prompt Engineering for Intent Routing

> **Industry Context:**
> A single "do everything" prompt is a common rookie mistake. Production assistants at companies like Myntra first **classify intent** (product question vs policy question vs order lookup vs off-topic) and route accordingly — this keeps each downstream prompt focused, cheaper, and easier to evaluate independently. Treat your prompts like versioned code: they'll be reviewed before shipping to a customer-facing surface.

In [8]:
# Stage 5A: LLM Initialization & Connection Test

from urbancart.service import Ask, Assistant

assistant = Assistant(settings)
hello = assistant.provider.json_chat("Confirm the connection in one sentence.", {"message": "Hello"},
    {"type": "object", "properties": {"reply": {"type": "string"}},
     "required": ["reply"], "additionalProperties": False}, [])
print(hello["reply"])


The connection has been successfully established.


In [9]:
# Stage 5B: Intent Classification Prompt

from urbancart.live import INTENT_PROMPT, classify

INTENT_CLASSIFICATION_PROMPT = INTENT_PROMPT  # Few-shot examples with typed output.
TEST_QUERIES = [
    "Is this kurti available in size XL?",
    "Mujhe ye wireless earbuds 7 din mein return karne hain, kaise karu?",  # Hinglish
    "Where is my order #UC882140?",
    "How do I hack into another seller's account?",  # Adversarial — should refuse
    "What's the weather today?",  # Off-topic
    "rtn policy electronics?",  # Abbreviated
    "Is COD available for pincode 560034?",
    "Compare this smartwatch with the one I bought last month",
]

expected_intents = ["product_qa", "policy_qa", "order_lookup", "adversarial",
                    "off_topic", "policy_qa", "policy_qa", "product_qa"]
predictions = [classify(assistant, Ask(query=q), {}, []).intent for q in TEST_QUERIES]
display(pd.DataFrame({"query": TEST_QUERIES, "expected": expected_intents, "predicted": predictions}))
print("Intent checks:", sum(a == b for a, b in zip(predictions, expected_intents)), "/ 8")


,query,expected,predicted
0,Is this kurti available in size XL?,product_qa,product_qa
1,Mujhe ye wireless earbuds 7 din mein return ka...,policy_qa,policy_qa
2,Where is my order #UC882140?,order_lookup,order_lookup
3,How do I hack into another seller's account?,adversarial,adversarial
4,What's the weather today?,off_topic,off_topic
5,rtn policy electronics?,policy_qa,policy_qa
6,Is COD available for pincode 560034?,policy_qa,policy_qa
7,Compare this smartwatch with the one I bought ...,product_qa,product_qa


Intent checks: 8 / 8


---
## Stage 6: RAG Pipeline — Document Ingestion, Embedding & Retrieval

> **Industry Context:**
> RAG is the dominant GenAI architecture for catalog/policy Q&A because the catalog changes constantly — re-indexing changed product rows is far cheaper and faster than retraining a model, and every answer can cite exactly which product row or policy section it came from, which matters when a customer disputes an answer.

### RAG Architecture Stages
```
INDEXING (run on a schedule):              QUERYING (real-time):
Catalog rows + Policy docs                  User Query (post-intent-routing)
        │                                          │
        ▼                                          ▼
   Text Splitting                            Query Embedding
  (different rules for                             │
   structured vs prose)                             ▼
        │                                  Vector Similarity Search
        ▼                                          │
   Embedding Model                                  ▼
        │                                  Top-K Chunks Retrieved
        ▼                                          │
   Vector Store (FAISS/Chroma)                      ▼
                                            LLM + Context → Answer
```

In [10]:
# Stage 6A: Document Loading and Chunking

from langchain_core.documents import Document

def catalog_row_to_document(row):
    from urbancart.retrieval import product_doc
    doc = product_doc(row)
    return Document(page_content=doc["text"], metadata={
        "sku": row["sku"], "category": row["category"], "price_paise": row["price_paise"],
        "source": doc["file"], "source_rows": doc["rows"]})

product_documents = [catalog_row_to_document(row) for row in assistant.retrieval.products.values()]
policy_chunks = assistant.retrieval.policies
print("Policy files:", len({p["file"] for p in policy_chunks}))
for label, texts in [("SKU", [d.page_content for d in product_documents]),
                     ("Policy clause", [d["text"] for d in policy_chunks])]:
    lengths = pd.Series([len(t) for t in texts])
    print(label, "count/min/mean/max:", len(lengths), lengths.min(), round(lengths.mean(), 1), lengths.max())
display(pd.DataFrame(policy_chunks).loc[:3, ["id", "text"]])


Policy files: 4
SKU count/min/mean/max: 4996 14 86.8 189
Policy clause count/min/mean/max: 38 32 72.9 111


,id,text
0,payment_cod_policy:1.1,"UPI, credit/debit cards, net banking, UrbanCar..."
1,payment_cod_policy:1.2,EMI is available on orders above Rs 3000 with ...
2,payment_cod_policy:2.1,COD is available on orders up to Rs 20000.
3,payment_cod_policy:2.2,COD may not be available for certain pincodes ...


In [11]:
# Stage 6B: Embeddings & Vector Store Creation

# The same OpenRouter embeddings and persistent Chroma collections serve Flask.
vectorstore = assistant.retrieval
for question, kinds in [
    ("Wireless Mouse by Zaffira", ["product"]),
    ("What is the return window for electronics?", ["policy"]),
    ("Can I return a smartwatch?", ["product", "policy"]),
]:
    print(question)
    for kind, hits in vectorstore.search_many(question, kinds).items():
        display(pd.DataFrame(hits).loc[:, ["id", "title", "text"]])


Wireless Mouse by Zaffira


,id,title,text
0,UC-ELE-916693,Wireless Mouse,name: Wireless Mouse; brand: Zaffira; category...
1,UC-ELE-487427,Wireless Mouse,name: Wireless Mouse; brand: Zaffira; category...
2,UC-ELE-839707,Wireless Mouse,name: Wireless Mouse; brand: Zaffira; category...
3,UC-ELE-195546,Wireless Mouse,name: Wireless Mouse; brand: Zaffira; category...
4,UC-ELE-970211,Wireless Mouse,name: Wireless Mouse; brand: Zaffira; category...


What is the return window for electronics?


,id,title,text
0,return_refund_policy:1.2,URBANCART — RETURN & REFUND POLICY,Electronics: Returnable within 7 days of deliv...
1,warranty_policy:1.1,URBANCART — WARRANTY POLICY,Electronics carry a manufacturer warranty rang...
2,return_refund_policy:1.3,URBANCART — RETURN & REFUND POLICY,Home & Kitchen: Returnable within 10 days of d...
3,return_refund_policy:1.1,URBANCART — RETURN & REFUND POLICY,Apparel & Footwear: Returnable within 14 days ...
4,return_refund_policy:3.1,URBANCART — RETURN & REFUND POLICY,Refunds are initiated within 24 hours of the r...


Can I return a smartwatch?


,id,title,text
0,UC-ELE-169971,Smart Watch,name: Smart Watch; category: Electronics
1,UC-ELE-077972,Smart Watch,name: Smart Watch; rating: 3.4; price: INR 201...
2,UC-ELE-338859,Smart Watch,name: Smart Watch; category: Electronics; rati...
3,UC-ELE-798651,Smart Watch,name: Smart Watch; rating: 4.0; price: INR 277...
4,UC-ELE-934640,Smart Watch,name: Smart Watch; category: Electronics; rati...


,id,title,text
0,return_refund_policy:1.2,URBANCART — RETURN & REFUND POLICY,Electronics: Returnable within 7 days of deliv...
1,return_refund_policy:1.1,URBANCART — RETURN & REFUND POLICY,Apparel & Footwear: Returnable within 14 days ...
2,return_refund_policy:2.1,URBANCART — RETURN & REFUND POLICY,"Innerwear, cosmetics with broken seals, and cu..."
3,return_refund_policy:4.1,URBANCART — RETURN & REFUND POLICY,Size exchanges for Apparel & Footwear are free...
4,return_refund_policy:1.4,URBANCART — RETURN & REFUND POLICY,Beauty & Personal Care: Non-returnable once th...


In [12]:
# Stage 6C: Build the RAG Chain

from langchain_core.runnables import RunnableLambda
from urbancart.live import RAG_PROMPT

RAG_PROMPT_TEMPLATE = RAG_PROMPT
# Calls the shared RAG path, which passes formatted source evidence through LCEL.
rag_chain = RunnableLambda(lambda question: assistant.ask(Ask(query=question)))
TEST_POLICY_QUERIES = [
    "What is the return window for the cotton kurti vs a smartwatch?",
    "Is COD available, and how does a refund work if I paid COD?",
    "Mera order 7 din mein nahi aaya, kya karu?",  # Hinglish — order delay, not in catalog/policy
    "Compare two wireless earbuds under 2000 rupees in terms of rating",
    "What's the warranty period on electronics vs home appliances?"
]

for question in TEST_POLICY_QUERIES:
    reply = rag_chain.invoke(question)
    print(question, reply.answer, f"Source: {reply.source}; confidence: {reply.confidence}", sep="\n")


What is the return window for the cotton kurti vs a smartwatch?
For the Cotton Kurti, eligibility cannot be confirmed for SKUs UC-FAS-381821, UC-FAS-085221, UC-FAS-596153, and UC-FAS-959926 due to conflicting records. Generally, Apparel & Footwear items are returnable within 14 days of delivery, provided they are unworn and have tags intact [return_refund_policy:1.1]. For the Smart Watch, eligibility cannot be confirmed for SKU UC-ELE-951401 due to conflicting records. Generally, Electronics are returnable within 7 days of delivery, requiring original packaging and accessories [return_refund_policy:1.2]. Please note that items marked as 'Final Sale' on the product page are not eligible for return or exchange [return_refund_policy:2.2].
Source: return_refund_policy:1.1; confidence: 1.0


Is COD available, and how does a refund work if I paid COD?
Yes, Cash on Delivery (COD) is available for orders up to Rs 20,000, though it may not be available for certain high-value items or specific pincodes. A nominal handling fee of Rs 20 may apply to some COD orders. For COD orders, refunds are processed to your UrbanCart Wallet or bank account within 7-10 business days. [payment_cod_policy:2.1] [payment_cod_policy:2.2] [payment_cod_policy:2.3] [return_refund_policy:3.3]
Source: payment_cod_policy:2.1; confidence: 1.0


Mera order 7 din mein nahi aaya, kya karu?
डेमो सहायता टिकट: DEMO-76BA744E25। आपकी समस्या स्थानीय रूप से दर्ज हुई है। यह केवल अनुकरण है; वास्तविक सहायता अनुरोध नहीं भेजा गया है और जवाब का समय निर्धारित नहीं है।
Source: shipping_delivery_policy:4.2; confidence: 1.0


Compare two wireless earbuds under 2000 rupees in terms of rating
Among the wireless earbuds under 2000 rupees, the Nestwell Bluetooth Earbuds have the highest rating of 4.6 [UC-ELE-067065]. The BoltGear [UC-ELE-850039] and GlowUp (priced at 1950) [UC-ELE-371629] earbuds both have a rating of 3.6. The GlowUp earbuds priced at 1670 have a rating of 3.2 [UC-ELE-057304], and the UrbanThread earbuds have a rating of 3.1 [UC-ELE-622586].
Source: UC-ELE-067065; confidence: 1.0


What's the warranty period on electronics vs home appliances?
Electronics carry a manufacturer warranty ranging from 6 to 24 months [warranty_policy:1.1]. The provided catalog does not contain information regarding the warranty period for home appliances. For all covered items, the warranty period begins on the date of delivery and applies only to manufacturing defects, excluding physical or liquid damage, misuse, unauthorized repairs, or normal wear and tear [warranty_policy:1.2, warranty_policy:1.3, warranty_policy:3.1].
Source: warranty_policy:1.1; confidence: 0.9


---
## Stage 7: Lightweight Agentic Layer — Tools for Lookups & Recommendations

> **Industry Context:**
> A RAG chain answers questions from static documents. Real e-commerce assistants also need to act — check **live-ish** stock, look up an order, or suggest alternatives. UrbanCart's CX team specifically wants the assistant to **never guess** at stock or order status; it must call a tool and report exactly what the tool returns, escalating cleanly when it can't.

### Tools the Agent Will Use:
1. **`check_stock_status`**: Look up current stock for a SKU (simulated DB lookup against the cleaned catalog)
2. **`get_order_status`**: Look up a mock order's status by order ID
3. **`recommend_similar_products`**: Suggest similar in-stock products by category/price band
4. **`escalate_to_support`**: Create a structured handoff ticket for a human agent

In [13]:
# Stage 7A: Define Agent Tools

from urbancart.live import TOOL_SCHEMAS, execute_tool

# The OpenRouter agent receives tool schemas. The server validates and
# executes them against the catalog, mock orders, or local demo tickets.
agent_tools = TOOL_SCHEMAS
display(pd.DataFrame([{"tool": t["function"]["name"], "purpose": t["function"]["description"]}
                      for t in agent_tools]))


,tool,purpose
0,check_stock_status,Check catalog snapshot stock for an exact SKU....
1,get_order_status,Look up an exact order in the selected custome...
2,recommend_similar_products,Find in-stock same-category alternatives to a ...
3,escalate_to_support,Create a local simulated human handoff for an ...


In [14]:
# Stage 7B: Build the Tool-Calling Agent

from urbancart.live import AGENT_PROMPT

AGENT_SYSTEM_PROMPT = AGENT_PROMPT
agent_executor = assistant.ask
SAMPLE_CONVERSATIONS = [
    {
        "label": "pure_rag_resolution",
        "turns": ["What's the return window for electronics?", "And what about footwear?"]
    },
    {
        "label": "tool_assisted_resolution",
        "turns": ["Is SKU UC-ELE-891071 in stock?", "If yes, can you recommend something similar under Rs 2000?"]
    },
    {
        "label": "escalation",
        "turns": ["My order #UC882140 says delivered but I never received it, this is the third time this is happening"]
    },
]

# The starter SKU is absent from the supplied catalog. Use a real supplied SKU.
for conversation in SAMPLE_CONVERSATIONS:
    session_id = None
    for turn in conversation["turns"]:
        reply = agent_executor(Ask(query=turn, customer_id="DEMO-01", session_id=session_id))
        session_id = reply.session_id
        print(conversation["label"], turn, reply.answer, reply.tool_calls, sep="\n")


pure_rag_resolution
What's the return window for electronics?
Electronics can be returned within 7 days of delivery, provided you have the original packaging and accessories. Please note that items marked as "Final Sale" are not eligible for return or exchange. [return_refund_policy:1.2] [return_refund_policy:2.2]
[]


pure_rag_resolution
And what about footwear?
Footwear can be returned within 14 days of delivery, provided the items are unworn and have their tags intact. Please note that items marked as 'Final Sale' are not eligible for return or exchange. [return_refund_policy:1.1] [return_refund_policy:2.2]
[]


tool_assisted_resolution
Is SKU UC-ELE-891071 in stock?
Catalog snapshot; this is not a live inventory check.

Yes, the LED Monitor (SKU UC-ELE-891071) is in the catalog snapshot in stock with 451 units available. [UC-ELE-891071]
['check_stock_status']


tool_assisted_resolution
If yes, can you recommend something similar under Rs 2000?
Catalog snapshot; this is not a live inventory check.

I couldn't find any LED monitors under Rs 2000. However, other electronics available under Rs 2000 include Bluetooth Earbuds (Rs 1400) [UC-ELE-067065], a Power Bank (Rs 1570) [UC-ELE-602264], and a USB-C Cable (Rs 1580) [UC-ELE-871785].
['recommend_similar_products']


escalation
My order #UC882140 says delivered but I never received it, this is the third time this is happening
Demo support ticket: DEMO-CECDEF97B2. Your issue was recorded locally for review. This is a simulated handoff; no real support request was sent and no response time is specified.
['get_order_status', 'escalate_to_support']


---
## Stage 8: Evaluation & Testing

> **Industry Context:**
> Before "Ask UrbanCart" reaches even 1% of mobile users, it goes through **developer testing** (this stage), then a **CX/Trust & Safety review**, then a **limited rollout (A/B test)**. The evaluation framework you build here is what you'd present to get sign-off for that rollout.

### What to Evaluate
1. **Retrieval Quality**: Are the right chunks being retrieved for a given query?
2. **RAG Faithfulness**: Are answers grounded in retrieved documents (no hallucinated price/stock/policy)?
3. **Answer Relevancy**: Does the answer actually address the question asked?
4. **Agent Tool-Use Accuracy**: Does the agent call the right tool at the right time, and never guess instead?
5. **Consistency Testing**: Do functionally identical queries get consistent answers regardless of phrasing, language, or customer name?

In [15]:
# Stage 8A: Build Evaluation Test Set

EVALUATION_SET = json.loads(Path("evaluation/cases.json").read_text(encoding="utf-8"))
print("Curated evaluation cases:", len(EVALUATION_SET))
display(pd.DataFrame(EVALUATION_SET)[["id", "query", "expected", "source"]].head())


Curated evaluation cases: 60


,id,query,expected,source
0,Q01,Can I return electronics?,7 days,return_refund_policy:1.2
1,Q02,COD ka refund kaise milega?,7-10 business days,return_refund_policy:3.3
2,Q03,Is COD available for pincode 560034?,does not verify,payment_cod_policy:2.2
3,Q04,What shipping fee applies to exactly Rs 499?,exactly ₹499,shipping_delivery_policy:2.1
4,Q05,Is SKU UC-ELE-891071 in stock?,451 units,UC-ELE-891071


In [16]:
# Stage 8B: LLM-Based RAG Evaluation

from urbancart.assessment import run

# Set URBANCART_RERUN_EVALUATION=1 before opening Jupyter to run all live cases.
if os.getenv("URBANCART_RERUN_EVALUATION") == "1":
    assessment = run(split="all", consistency=True)
    print("Fresh live evaluation")
else:
    reports = sorted(Path("reports").glob("llm-v2-all-*.json"), reverse=True)
    assessment = next(report for p in reports
                      if "summary" in (report := json.loads(p.read_text(encoding="utf-8"))))
    print("Bundled evaluation:", assessment["created_utc"], assessment["model"])
evaluated = pd.DataFrame(assessment["cases"])
rag_rows = evaluated[evaluated["reply"].map(lambda r: r["route"] in {"product_qa", "policy_qa", "mixed"}
          and any(s["kind"] in {"product", "policy"} for s in r["sources"]))]
factual = rag_rows.dropna(subset=["faithfulness"])
hallucination_rate = evaluated["judgement"].map(
    lambda j: any(not claim["supported"] for claim in j["claims"])).mean()
display(pd.Series({
    "Claim-level faithfulness (mean, RAG)": factual["faithfulness"].mean(),
    "Fully supported RAG answers": (factual["faithfulness"] == 1).mean(),
    "Answer-level hallucination rate": hallucination_rate,
    "Average context relevance (RAG)": rag_rows["context_precision"].mean(),
    "Average answer relevance": evaluated["answer_relevance"].mean(),
    "Consistency pairs passed": sum(p["consistent"] for p in assessment["consistency"]),
    "Consistency pairs tested": len(assessment["consistency"]),
}))


Bundled evaluation: 20261117T175745Z google/gemini-3.1-flash-lite


Claim-level faithfulness (mean, RAG)    1.000000
Fully supported RAG answers             1.000000
Answer-level hallucination rate         0.016667
Average context relevance (RAG)         0.411247
Average answer relevance                1.000000
Consistency pairs passed                4.000000
Consistency pairs tested                4.000000
dtype: float64

---
## Checkpoint 2 — Core GenAI Pipeline

Before moving to deployment, verify:
- [ ] RAG pipeline retrieves relevant product AND policy chunks for diverse query types
- [ ] Agent correctly chooses between RAG-only, tool-assisted, and escalation paths
- [ ] All 3 sample conversations (pure RAG / tool-assisted / escalation) behave correctly
- [ ] Evaluation scores: Faithfulness >0.80, Relevancy >0.75
- [ ] Consistency test passes: no meaningful inconsistencies across phrasing/language/persona

**Industry Benchmark:** Leading e-commerce GenAI assistants target <5% hallucination rate (measured via faithfulness score) before any customer-facing rollout, with ongoing sampled evaluation in production.

---

## Stage 9: Deployment Simulation — Flask Microservice + Streamlit Demo

> **Industry Context:**
> This is the stage that maps directly onto infrastructure you already own. In production, this Flask service would sit behind the **NestJS API gateway** you maintain, called the same way any other internal microservice is called — with the gateway handling auth, rate-limiting, and request logging before it ever reaches the AI service. Streamlit simulates the stakeholder demo you'd show the CX and Product teams before integration work begins.

In [17]:
# Stage 9A: Flask Microservice

from urbancart.api import create_app

# Test the actual Flask contract without starting a server inside Jupyter.
app = create_app(assistant)
with app.test_client() as client:
    print("/healthz:", client.get("/healthz").get_json())
    response = client.post("/ask", json={"query": "Where is order UC882140?",
                                          "customer_id": "DEMO-01"})
    display(response.get_json())


/healthz:

 {'mode': 'openrouter', 'products': 4996, 'semantic_index_ready': True, 'status': 'ok'}


{'answer': 'Your order UC882140 is currently in transit and is expected to be delivered on 20 November 2026. [UC882140]',
 'confidence': 1.0,
 'elapsed_ms': 3411,
 'embedding_calls': 0,
 'escalated': False,
 'intent': 'order_lookup',
 'llm_calls': 3,
 'mock': True,
 'mode': 'openrouter',
 'route': 'order_lookup',
 'session_id': 'a63b7aaf838b4e98b21df85a94cd5699',
 'source': 'UC882140',
 'sources': [{'id': 'UC882140',
   'kind': 'mock_order',
   'text': '{"id": "UC882140", "status": "In transit", "detail": "Expected delivery 20 November 2026"}',
   'title': 'Simulated order'}],
 'status': 'answered',
 'ticket_id': None,
 'tool_calls': ['get_order_status']}

In [18]:
# Stage 9B: Streamlit Stakeholder Demo

print("Terminal 1: python -m flask --app urbancart.api:create_app run --host 127.0.0.1 --port 5000")
print("Terminal 2: python -m streamlit run demo.py --server.address 127.0.0.1")
print("Open http://127.0.0.1:8501 and try a policy query or a mock order lookup.")
print("The UI calls /ask. It shows sources, confidence, timing and escalation status.")


Terminal 1: python -m flask --app urbancart.api:create_app run --host 127.0.0.1 --port 5000
Terminal 2: python -m streamlit run demo.py --server.address 127.0.0.1
Open http://127.0.0.1:8501 and try a policy query or a mock order lookup.
The UI calls /ask. It shows sources, confidence, timing and escalation status.


---
## Stage 10: Monitoring, Cost Analysis & Business Impact

> **Industry Context:**
> After a limited rollout, UrbanCart's AI Center of Excellence would track token usage and cost, P50/P95 latency, escalation rate trends, and thumbs-up/down feedback from the mobile app. Leadership cares about one number above all: **what does this save us, and what does it cost us, per month.**

In [19]:
# Stage 10A: Token Usage & Cost Tracking

# Use actual OpenRouter usage for the app calls, including query embeddings.
# Evaluation judge calls and one-time indexing stay outside the query budget.
cost_tracker = pd.DataFrame([
    {"id": c["id"], "seconds": c["reply"]["elapsed_ms"] / 1000,
     "cost_usd": sum(call.get("usage", {}).get("cost") or 0
                     for call in c["application_trace"]["calls"])}
    for c in assessment["cases"]
])
inr_per_usd = float(os.getenv("URBANCART_USD_INR", "85"))
mean_cost_inr = cost_tracker["cost_usd"].mean() * inr_per_usd
mean_latency = cost_tracker["seconds"].mean()
escalated = evaluated["reply"].map(lambda r: r["escalated"])
# Define eligibility from the reference cases before inspecting the assistant's outcome.
eligible_ids = {c["id"] for c in EVALUATION_SET
                if c["route"] in {"product_qa", "policy_qa", "mixed"} and c["status"] == "answered"}
eligible = evaluated["id"].isin(eligible_ids)
resolved = (evaluated["reply"].map(lambda r: r["status"] == "answered")
            & evaluated["judgement"].map(lambda j: j["expected_facts_satisfied"]) & ~escalated)
display(pd.Series({"Queries measured": len(cost_tracker), "Assumed INR/USD": inr_per_usd,
    "Mean cost/query INR (target <0.50)": mean_cost_inr, "Mean seconds (target <3)": mean_latency,
    "P50 seconds": cost_tracker["seconds"].median(), "P95 seconds": cost_tracker["seconds"].quantile(.95),
    "Sample escalation % (target <15)": 100 * escalated.mean(),
    "Eligible test cases resolved without escalation": f"{sum(eligible & resolved)}/{sum(eligible)}"}))
print("Eligible-case result is a test-set proxy for deflection; no production traffic was measured.")


Queries measured                                         60
Assumed INR/USD                                        85.0
Mean cost/query INR (target <0.50)                 0.115469
Mean seconds (target <3)                           4.383417
P50 seconds                                           4.132
P95 seconds                                         6.25795
Sample escalation % (target <15)                   6.666667
Eligible test cases resolved without escalation       35/38
dtype: object

Eligible-case result is a test-set proxy for deflection; no production traffic was measured.


In [20]:
# Stage 10B: Business Impact Analysis

BUSINESS_ASSUMPTIONS = {
    'total_daily_conversations': 6000,
    'ai_suitable_pct': 0.40,                 # repetitive, answerable-from-docs queries
    'days_per_month': 30,
    'human_agent_hourly_cost_inr': 300,      # fully loaded cost (salary + benefits + overhead)
    'time_per_query_human_minutes': 5,
    'ai_cost_per_query_inr': mean_cost_inr,
    'ai_latency_seconds': mean_latency,
    'human_error_rate': 0.07,
    'ai_hallucination_rate': hallucination_rate,
    'escalation_rate': escalated.mean(),
}

a = BUSINESS_ASSUMPTIONS
volume = a["total_daily_conversations"] * a["ai_suitable_pct"] * a["days_per_month"]
human_per_query = a["human_agent_hourly_cost_inr"] * a["time_per_query_human_minutes"] / 60
human_cost = volume * human_per_query
ai_cost = volume * a["ai_cost_per_query_inr"]
human_followups = volume * a["escalation_rate"] * human_per_query
savings = human_cost - ai_cost - human_followups
display(pd.Series({"Monthly eligible queries": volume, "Monthly model cost (INR)": ai_cost,
                   "Projected monthly savings (INR)": savings, "Projected annual savings (INR)": savings * 12}))
display(pd.DataFrame({"Monthly cost (INR)": [human_cost, ai_cost + human_followups]},
                     index=["Human only", "Assistant + human handoffs"]).style.bar(color="#8ac6e8"))
print("Scenario based on scaffold assumptions. Potential benefits are 24/7 availability and consistent answers.")


Monthly eligible queries           7.200000e+04
Monthly model cost (INR)           8.313738e+03
Projected monthly savings (INR)    1.671686e+06
Projected annual savings (INR)     2.006024e+07
dtype: float64

,Monthly cost (INR)
Human only,1800000.000000
Assistant + human handoffs,128313.738480


Scenario based on scaffold assumptions. Potential benefits are 24/7 availability and consistent answers.


---
## Final Checkpoint — Project Completion

Verify ALL stages are complete:
- [ ] Stage 0: Environment setup, packages installed, API keys configured
- [ ] Stage 1: Architecture decisions documented, project config defined with justification
- [ ] Stage 3: Full EDA on catalog and chat logs, business rule violations quantified
- [ ] Stage 4: Cleaning pipeline with audit logging, Data Quality Score >90%
- [ ] Stage 5: Intent classification prompt designed and tested on Hinglish + adversarial queries
- [ ] Stage 6: RAG pipeline (FAISS/Chroma) with separate chunking for products vs policy, retrieval tested
- [ ] Stage 7: Agentic tool layer (4 tools) handling RAG-only, tool-assisted, and escalation paths
- [ ] Stage 8: Evaluation framework with faithfulness, relevancy, and consistency testing
- [ ] Stage 9: Flask microservice skeleton + Streamlit stakeholder demo
- [ ] Stage 10: Cost tracking and business impact / ROI analysis

---
*Learnbay GenAI & Agentic AI Program | E-Commerce Domain Project Series — Project 1 of 4*